In [1]:
import json
import re
import pandas as pd
from pathlib import Path

In [2]:
def load_raw_data(path="raw_data/abcd_v1.1.json"):
    with open(path) as f:
        return json.load(f)

data = load_raw_data()
print({split: len(chats) for split, chats in data.items()})

{'train': 8034, 'dev': 1004, 'test': 1004}


In [3]:


OPENER_PATTERNS = re.compile(
    r"how (can|may|could|do) i (help|assist)"
    r"|what (can|may) i (do for you|help)"
    r"|how can i be of (help|assistance)"
    r"|^\s*(hi|hello|hey)( there)?[\s.,!]*$",
    re.IGNORECASE,
)

def is_opener(text, max_words=20):
    return len(text.split()) <= max_words and bool(OPENER_PATTERNS.search(text))

In [4]:
def extract_first_messages(turns, stop_on_action=True):
    parts = []
    for speaker, text in turns:
        if speaker == "customer":
            parts.append(text)
        elif speaker == "agent":
            if is_opener(text):
                continue
            if parts:
                break
        elif speaker == "action":
            if parts and stop_on_action:
                break
    return re.sub(r"\s+", " ", " ".join(parts)).strip()

In [5]:
REFUND_TEXT = "refund has been made"

def has_refund(turns):
    return int(any(t[0] == "action" and REFUND_TEXT in t[1].lower() for t in turns))

rows = []
target_subflows = ("refund_initiate",
    "refund_update",
    "cost",
    "manage_dispute_bill",
    "manage_cancel",
    "status_service_added",
    "status_quantity")

for split in ["train", "dev", "test"]:
    for chat in data[split]:
        turns = chat["original"]
        subflow = chat.get("scenario", {}).get("subflow", "")
        label = 1 if subflow in target_subflows else 0
        rows.append({
            "split": split,
            "id": chat["convo_id"],
            "flow": chat["scenario"].get("flow"),
            "subflow": chat["scenario"].get("subflow"),
            "first_msgs": extract_first_messages(turns),
            "refund": has_refund(turns),
            'refund_label': label
        })

abcd = pd.DataFrame(rows)
abcd["n_words"] = abcd["first_msgs"].str.split().str.len()
abcd.shape

(10042, 8)

In [6]:
abcd.head(30)

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,0,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,0,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,0,34
5,train,1426,subscription_inquiry,manage_extension,I want an extension for my shopping service. I...,0,0,16
6,train,8893,account_access,recover_username,"Hello, I am trying to check the status of my o...",0,0,26
7,train,5185,troubleshoot_site,shopping_cart,Hello I am trying to buy a pair of boots but t...,0,0,18
8,train,10015,shipping_issue,manage,I accidently ordered the wrong item and was wo...,1,0,15
9,train,1884,shipping_issue,status,I would like to confirm my address is correct ...,0,0,20


In [7]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
abcd.to_parquet("data/processed/abcd_first_msgs.parquet", index=False)